# Download e preparazione di VideoXum su Leonardo

Il notebook scarica i video ActivityNet e li organizza secondo gli split ufficiali di `jylins/videoxum`: `train/`, `val/` e `test/`. I dati finali sono salvati per default in `$WORK/$USER/datasets/videoxum`; cache e archivi temporanei vanno in `$SCRATCH/videoxum_download`.

Eseguire in un job Slurm, non sul login node: il trasferimento è lungo e richiede centinaia di GB. Il processo è riavviabile e salta i video già estratti.


In [1]:
import json
import os
import re
import shutil
import tarfile
import zipfile
from getpass import getuser
from pathlib import Path

from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm

# Configurazione Leonardo. Si possono sovrascrivere i due percorsi con variabili ambiente.
user = os.environ.get("USER", getuser())
work_root = os.environ.get("WORK")
scratch_root = os.environ.get("SCRATCH") or os.environ.get("CINECA_SCRATCH")
if not work_root:
    raise RuntimeError("$WORK non definita: impostare VIDEOXUM_DATA_ROOT")
if not scratch_root:
    raise RuntimeError("$SCRATCH non definita: impostare VIDEOXUM_TMP_ROOT")

DATA_ROOT = Path(os.environ.get(
    "VIDEOXUM_DATA_ROOT", str(Path(work_root) / user / "datasets" / "videoxum")
)).expanduser().resolve()
TMP_ROOT = Path(os.environ.get(
    "VIDEOXUM_TMP_ROOT", str(Path(scratch_root) / "videoxum_download")
)).expanduser().resolve()
HF_CACHE = TMP_ROOT / "hf_cache"
ARCHIVE_ROOT = TMP_ROOT / "activitynet_archives"

# None = tutti i 14.001 video. Usare 230.0 per mantenere il precedente filtro.
MIN_DURATION = None
KEEP_ARCHIVES = False
SPLIT_TO_DIR = {"train": "train", "validation": "val", "test": "test"}
VIDEO_EXTENSIONS = {".mp4", ".mkv", ".webm", ".avi", ".mov", ".m4v"}
ACTIVITYNET_REPO = "YimuWang/ActivityNet"
ARCHIVE_FAMILIES = ("v1-2_train", "v1-2_val", "v1-3_train_val")

for directory in [
    *(DATA_ROOT / split for split in SPLIT_TO_DIR.values()),
    DATA_ROOT / "metadata", HF_CACHE, ARCHIVE_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Dati finali: {DATA_ROOT}")
print(f"Area temporanea: {TMP_ROOT}")
print("Caricamento degli split ufficiali jylins/videoxum...")
dataset = load_dataset("jylins/videoxum", cache_dir=str(HF_CACHE))
records_by_split = {}
ids_by_split = {}
for hf_split, output_split in SPLIT_TO_DIR.items():
    records = [
        dict(row) for row in dataset[hf_split]
        if MIN_DURATION is None or float(row["duration"]) >= MIN_DURATION
    ]
    records_by_split[output_split] = records
    ids_by_split[output_split] = {row["video_id"] for row in records}
    print(f"{output_split:>5}: {len(records):>5} video")

all_ids = set().union(*ids_by_split.values())
if len(all_ids) != sum(map(len, ids_by_split.values())):
    raise RuntimeError("Uno stesso video_id compare in più split VideoXum")
id_to_split = {
    video_id: split for split, video_ids in ids_by_split.items() for video_id in video_ids
}


def existing_video_ids():
    return {
        path.stem
        for split in SPLIT_TO_DIR.values()
        for path in (DATA_ROOT / split).iterdir()
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
    }


def destination_for(member_name):
    source = Path(member_name)
    video_id = source.stem
    split = id_to_split.get(video_id)
    suffix = source.suffix.lower()
    if split is None or suffix not in VIDEO_EXTENSIONS:
        return None
    return video_id, DATA_ROOT / split / f"{video_id}{suffix}"


def copy_selected(source_file, member_name, pending):
    match = destination_for(member_name)
    if match is None or match[0] not in pending:
        return False
    video_id, destination = match
    temporary = destination.with_suffix(destination.suffix + ".part")
    try:
        with temporary.open("wb") as target:
            shutil.copyfileobj(source_file, target, length=16 * 1024 * 1024)
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)
    pending.remove(video_id)
    return True


class MultipartReader:
    """Presenta gli shard .00, .01, ... come un singolo stream."""
    def __init__(self, paths):
        self.paths = iter(paths)
        self.current = None

    def read(self, size=-1):
        if size == 0:
            return b""
        chunks, remaining = [], size
        while size < 0 or remaining > 0:
            if self.current is None:
                try:
                    self.current = next(self.paths).open("rb")
                except StopIteration:
                    break
            chunk = self.current.read(-1 if size < 0 else remaining)
            if chunk:
                chunks.append(chunk)
                if size > 0:
                    remaining -= len(chunk)
            else:
                self.current.close()
                self.current = None
        return b"".join(chunks)

    def close(self):
        if self.current is not None:
            self.current.close()
            self.current = None


def extract_tar_parts(paths, pending):
    copied = 0
    reader = MultipartReader(paths)
    try:
        with tarfile.open(fileobj=reader, mode="r|gz") as archive:
            for member in archive:
                if not member.isfile():
                    continue
                match = destination_for(member.name)
                if match is None or match[0] not in pending:
                    continue
                source = archive.extractfile(member)
                if source is not None:
                    with source:
                        copied += int(copy_selected(source, member.name, pending))
    finally:
        reader.close()
    return copied


def extract_zip(path, pending):
    copied = 0
    with zipfile.ZipFile(path) as archive:
        for member in tqdm(archive.infolist(), desc=f"Estrazione {path.name}"):
            if member.is_dir():
                continue
            match = destination_for(member.filename)
            if match is None or match[0] not in pending:
                continue
            with archive.open(member) as source:
                copied += int(copy_selected(source, member.filename, pending))
    return copied


repo_files = HfApi().list_repo_files(ACTIVITYNET_REPO, repo_type="dataset")
parts_by_family = {}
for family in ARCHIVE_FAMILIES:
    pattern = re.compile(rf"^{re.escape(family)}\.tar\.gz\.(\d+)$")
    parts = [(int(match.group(1)), name) for name in repo_files if (match := pattern.match(name))]
    parts_by_family[family] = [name for _, name in sorted(parts)]
    if not parts_by_family[family]:
        raise RuntimeError(f"Nessuno shard trovato per {family}")

pending = all_ids - existing_video_ids()
print(f"Già presenti: {len(all_ids) - len(pending)}; da recuperare: {len(pending)}")
for family, filenames in parts_by_family.items():
    if not pending:
        break
    family_dir = ARCHIVE_ROOT / family
    family_dir.mkdir(parents=True, exist_ok=True)
    print(f"\nDownload di {family}: {len(filenames)} shard")
    local_parts = [Path(hf_hub_download(
        repo_id=ACTIVITYNET_REPO,
        filename=filename,
        repo_type="dataset",
        local_dir=family_dir,
        cache_dir=HF_CACHE,
    )) for filename in filenames]
    before = len(pending)
    copied = extract_tar_parts(local_parts, pending)
    print(f"Estratti da {family}: {copied}; ancora mancanti: {len(pending)}")
    if before - len(pending) != copied:
        raise RuntimeError("Conteggio interno incoerente durante estrazione")
    if not KEEP_ARCHIVES:
        for path in local_parts:
            path.unlink(missing_ok=True)

# Integra i video train/validation non più disponibili negli archivi principali.
if pending and "missing_files.zip" in repo_files:
    print("\nDownload archivio dei video mancanti...")
    missing_archive = Path(hf_hub_download(
        repo_id=ACTIVITYNET_REPO,
        filename="missing_files.zip",
        repo_type="dataset",
        local_dir=ARCHIVE_ROOT / "missing_files",
        cache_dir=HF_CACHE,
    ))
    copied = extract_zip(missing_archive, pending)
    print(f"Estratti da missing_files.zip: {copied}; ancora mancanti: {len(pending)}")
    if not KEEP_ARCHIVES:
        missing_archive.unlink(missing_ok=True)

# Metadati separati per split e verifica finale.
combined_metadata = []
summary = {}
for split, records in records_by_split.items():
    videos = {
        path.stem: path for path in (DATA_ROOT / split).iterdir()
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
    }
    split_metadata, missing_ids = [], []
    for row in records:
        video_id = row["video_id"]
        video_path = videos.get(video_id)
        if video_path is None:
            missing_ids.append(video_id)
            continue
        entry = {
            "id": video_id,
            "split": split,
            "video_file": str(video_path.relative_to(DATA_ROOT)),
            "duration": float(row["duration"]),
            "ground_truth": row["tsum"],
        }
        split_metadata.append(entry)
        combined_metadata.append(entry)
    with (DATA_ROOT / "metadata" / f"{split}.json").open("w", encoding="utf-8") as handle:
        json.dump(split_metadata, handle, indent=2, ensure_ascii=False)
    with (DATA_ROOT / "metadata" / f"missing_{split}.txt").open("w", encoding="utf-8") as handle:
        handle.write("\n".join(sorted(missing_ids)))
    summary[split] = {
        "attesi": len(records), "presenti": len(split_metadata), "mancanti": len(missing_ids)
    }

with (DATA_ROOT / "metadata" / "all.json").open("w", encoding="utf-8") as handle:
    json.dump(combined_metadata, handle, indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f"Dataset pronto in: {DATA_ROOT}")
if any(item["mancanti"] for item in summary.values()):
    print("ATTENZIONE: controllare metadata/missing_<split>.txt")


/leonardo/home/userexternal/aintini0/GRPOVideo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dati finali: /leonardo_work/IscrC_MSMU/aintini0/datasets/videoxum
Area temporanea: /leonardo_scratch/large/userexternal/aintini0/videoxum_download
Caricamento degli split ufficiali jylins/videoxum...


Generating test split: 100%|██████████| 4000/4000 [00:01<00:00, 2320.66 examples/s]


train:  8000 video
  val:  2001 video
 test:  4000 video
Già presenti: 0; da recuperare: 14001

Download di v1-2_train: 29 shard


: 

## Configurazione

Per conservare il precedente filtro sui video lunghi almeno 230 secondi, impostare `MIN_DURATION = 230.0`. Per cambiare storage usare `VIDEOXUM_DATA_ROOT` e `VIDEOXUM_TMP_ROOT`. Gli eventuali ID non reperibili vengono salvati in `metadata/missing_<split>.txt`.
